# Supplementary Figure 13 | GWAS-QTL CoS properties and discovery

**a.** Distance from each QTL gene’s transcription start site (TSS) to the nearest ColocBoost CoS variant for GWAS-QTL CoS (GWAS p < 10-5). Percentages are of genes whose nearest CoS variant is within (91.1%), or outside of (8.9%) the cis window (TSS ± 1 Mb). QTL phenotypes from genes with no CoS variant within TSS ± 1 Mb were discarded from the CoS in further analysis to match a 1 Mb cis-window definition.

**b.** Distribution of the fraction of QTL phenotypes in each GWAS-QTL CoS with a CoS variant within TSS ± 1 Mb. Color indicates subthreshold GWAS (GWAS p < 10-5) and genome-wide significant GWAS (GWAS p < 5.7*10-9). 6.1% of GWAS-CoS have all QTL genes outside the cis window; these CoS were entirely discarded. For CoS where only some genes were outside the cis window, only phenotypes from those genes were discarded.

**c.** Fraction of QTL phenotypes in each GWAS-QTL CoS with any ColocBoost CoS variant in a SuSiE 95% credible set for that phenotype, with the same GWAS p-value cutoffs as (b). 69.3% of CoS show some SuSiE overlap, but many loci recovered by ColocBoost are not fine-mapped by SuSiE.

**d.** Per-tissue percentage of genes with a GWAS-QTL CoS (GWAS p < 10-5), by biotype. Dots are per-tissue single-tissue rates, colored by GTEx tissue. Outline bars are the rate across all tissues, with the edge colored by biotype. 4.2% of tested novel lncRNAs have a GWAS-eQTL CoS and 0.7% have a GWAS-sQTL CoS in some tissue.

**e.** Unique genes with a GWAS-QTL CoS (GWAS p < 10-5), by biotype and QTL modality. Bars are eQTL-only, both eQTL and sQTL, or sQTL-only.

**f.** Gene-level probability of a GWAS-QTL CoS (GWAS p < 10-5) versus maximum median expression in any tissue, maximum number of tested introns in any tissue, or number of tissues QTL-tested, stratified by biotype. Points are means in 20 equal-frequency bins; shaded bands are Wald 95% CIs for the binomial proportion. Colocalization rate increases with expression, intron number, and QTL ascertainment but is largely similar across biotypes at matched expression or tissue number.


## Setup

In [ ]:
import glob
import gzip
import re
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from tqdm.auto import tqdm

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 5.0)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

fig_width, fig_height, fig_dpi = 7.2, 5.0, 300

cis_bp = 1_000_000

gws_p_cut = 5.7e-9
gws_neglog10p = -np.log10(gws_p_cut)
gwas_p_cut_1e5 = 1e-5
gwas_neglog10p_1e5 = -np.log10(gwas_p_cut_1e5)
cos_npc_cut = 0.5
_ensg_stem = re.compile(r"(ENSG\d+)")
qtl_label_for_figure = {
    "no_qtl": "no_qtl",
    "non-lncRNA": "non-lncRNA",
    "Shared": "Shared",
    "lncRNA-specific (weak)": "Shared",
    "lncRNA-specific (robust)": "lncRNA-specific",
}
n_prob_bins = 20
min_bin_n = 25
bar_color = "#bdbdbd"  # p < 1e-5 (all)
gws_color = "#6e6e6e"  # GWS subset overlay

gene_type_order = ["novel_lncRNA", "existing_lncRNA", "protein_coding"]
gene_type_pretty = {
    "novel_lncRNA": "novel lncRNA",
    "existing_lncRNA": "existing lncRNA",
    "protein_coding": "protein coding",
}
_paired = sns.color_palette("Paired", 12)
gene_type_palette = {
    "novel_lncRNA": _paired[5],
    "existing_lncRNA": _paired[4],
    "protein_coding": _paired[1],
}

gene_modality_rows = [
    ("protein_coding", "protein-\ncoding"),
    ("novel_lncRNA", "novel\nlncRNA"),
    ("existing_lncRNA", "existing\nlncRNA"),
]
gene_modality_stack = ["eqtl", "both", "sqtl"]
gene_modality_label = {"eqtl": "eQTL", "both": "both", "sqtl": "sQTL"}
_eq_rgb = np.array([0x3B, 0x7E, 0xA1]) / 255.0
_sq_rgb = np.array([0xE8, 0xA8, 0x38]) / 255.0
_both_rgb = 0.5 * (_eq_rgb + _sq_rgb)
gene_modality_color = {
    "eqtl": "#3B7EA1",
    "both": "#%02x%02x%02x" % tuple(int(round(255 * x)) for x in _both_rgb),
    "sqtl": "#E8A838",
}


## Paths

In [ ]:
allpairs_example = Path("path/to/Adipose_Subcutaneous.v11.allpairs.chr1.parquet")
expr_bed = Path("path/to/Adipose_Subcutaneous.v11.normalized_expression.bed.gz")
qtl_tissue_ids_file = Path("path/to/qtl_tissue_ids.txt")

lead_glob_full_dist = "path/to/tissue_full_dist/*.joint_gwas.all.leads.parquet"
lead_glob = "path/to/*.joint_gwas.all.leads.parquet"
tss_gtf_v39 = Path("path/to/gencode.v39.genes.gtf")
tss_gtf_v47 = Path("path/to/gencode.v47.genes.gtf")
eqtl_susie_dir = Path("path/to/v11_eqtl_susie")
sqtl_susie_dir = Path("path/to/v11_sqtl_susie")
gencode_class = Path("path/to/gencode_v39_v47_gene_classification.txt")
cross_long = Path("path/to/cross_tissue_merged.long.parquet")
cross_summary = Path("path/to/cross_tissue_merged.summary.parquet")
median_counts_file = Path("path/to/tissue_median_counts.txt")
n_junctions_file = Path("path/to/gene_tissue_n_junctions.parquet")
egene_gt_file = Path("path/to/egene_gene_tissue.parquet")
sgene_gt_file = Path("path/to/sgene_gene_tissue.parquet")
gtex_colors_json = Path("refs/colors.json")

fig_dir = Path("figures")
fig_pdf = fig_dir / "supp_figure_13_colocboost.pdf"
fig_png = fig_dir / "supp_figure_13_colocboost.png"


## Load data

### Cis-window sanity check

`allpairs` `tss_distance` is clipped to `[-1e6, 1e6]` (TSS ± 1 Mb). Expression BED intervals are 1 bp at the TSS (spot-check only); panels a/b use GENCODE `genes.gtf` TSS to match the pipeline cis trim.

In [ ]:
ap = pd.read_parquet(allpairs_example, columns=["tss_distance"])
print(
    "allpairs tss_distance min/max:",
    int(ap["tss_distance"].min()),
    int(ap["tss_distance"].max()),
)

with gzip.open(expr_bed, "rt") as fh:
    header = fh.readline().rstrip("\n").split("\t")
    row = fh.readline().rstrip("\n").split("\t")
print("expression BED cols:", header[:4])
print("example gene row:", row[:4], "width=", int(row[2]) - int(row[1]))

### QTL tissue IDs

In [ ]:
qtl_tissue_ids = set(pd.read_table(qtl_tissue_ids_file, header=None)[0])

### GTEx tissue colors

In [ ]:
with open(gtex_colors_json) as f:
    _colors = json.load(f)
    tissue_df = pd.DataFrame.from_dict(_colors, orient="index").reset_index(names="tissue_name")
tissue_df["tissue_file_name"] = (
    tissue_df["tissue_name"]
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace(" - ", "_")
    .str.replace(" ", "_")
)
tissue_df["tissue_color_hex"] = "#" + tissue_df["tissue_color_hex"]
tissue_df = tissue_df[tissue_df["tissue_file_name"].isin(qtl_tissue_ids)].copy()
tissue_color = dict(zip(tissue_df["tissue_file_name"], tissue_df["tissue_color_hex"]))
print(f"QTL tissues with colors: {len(tissue_color)}")


### ColocBoost lead file lists


In [ ]:
lead_files_full_dist = sorted(glob.glob(lead_glob_full_dist))
lead_files = sorted(glob.glob(lead_glob))
if not lead_files_full_dist:
    raise FileNotFoundError(f"No full-dist lead files matching {lead_glob_full_dist}")
if not lead_files:
    raise FileNotFoundError(f"No tissue lead files matching {lead_glob}")
lead_files_dist = lead_files_full_dist
print(f"full-dist lead files (panels a/b): {len(lead_files_dist)}")
print(f"tissue lead files (panel c): {len(lead_files)}")


### CoS lead helpers


In [ ]:
def parse_variant_pos(variant_id: str):
    s = str(variant_id)
    if ":" in s:
        parts = s.split(":")
        return parts[0], int(parts[1])
    parts = s.split("_")
    return parts[0], int(parts[1])


def parse_neglog10_list(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []
    out = []
    for tok in str(val).split(","):
        tok = tok.strip()
        if not tok or tok.lower() == "nan":
            out.append(np.nan)
        else:
            try:
                out.append(float(tok))
            except ValueError:
                out.append(np.nan)
    return out


def has_gwas_gws(phenotype_id, cs_max_neg_log10_p, cut=gws_neglog10p) -> bool:
    parts = [p.strip() for p in str(phenotype_id).split(";") if p.strip()]
    vals = parse_neglog10_list(cs_max_neg_log10_p)
    for i, part in enumerate(parts):
        if i >= len(vals):
            break
        pl = part.lower()
        if "expression" in pl or "splicing" in pl:
            continue
        v = vals[i]
        if np.isfinite(v) and float(v) > cut:
            return True
    return False


def chrom_equal(a: str, b: str) -> bool:
    return str(a).replace("chr", "") == str(b).replace("chr", "")


def lead_variant_id(row):
    if "lead_variant_id" in row.index and pd.notna(row.get("lead_variant_id")):
        return str(row["lead_variant_id"])
    return str(row["variant_id"])


def robust_parquet_for_leads(lead_path: str) -> str:
    path = lead_path.replace(".leads.parquet", ".robust.parquet")
    if Path(path).is_file():
        return path
    raise FileNotFoundError(f"No robust parquet for {lead_path}")


def load_cos_variant_list_map(rob_path: str, wanted_coloc_ids: set):
    rob = pd.read_parquet(rob_path)
    if "coloc_id" not in rob.columns:
        rob["coloc_id"] = rob["filtered_gene"].astype(str) + "_" + rob["cs_id"].astype(str)
    rob = rob[rob["coloc_id"].astype(str).isin(wanted_coloc_ids)]
    return (
        rob.groupby("coloc_id")["variant_id"]
        .apply(lambda s: list(pd.unique(s.astype(str))))
        .to_dict()
    )


def gene_has_outside_cos(gene_id: str, variant_ids, tss_lookup, window=cis_bp):
    """True if ALL CoS variants are outside TSS±window (none within); None if TSS unknown."""
    key = gene_id if gene_id in tss_lookup else str(gene_id).split(".")[0]
    if key not in tss_lookup:
        return None
    chrom, tss = tss_lookup[key]
    for vid in variant_ids:
        vchrom, vpos = parse_variant_pos(vid)
        if chrom_equal(vchrom, chrom) and abs(vpos - tss) <= window:
            return False
    return True


def as_stem_set(vals):
    out = set()
    if vals is None:
        return out
    try:
        if isinstance(vals, float) and np.isnan(vals):
            return out
    except (TypeError, ValueError):
        pass
    items = vals if isinstance(vals, (list, tuple, np.ndarray, set)) else [vals]
    for v in items:
        if v is None:
            continue
        m = _ensg_stem.search(str(v))
        if m:
            out.add(m.group(1))
    return out


def as_str_list(vals):
    if vals is None:
        return []
    try:
        if isinstance(vals, float) and np.isnan(vals):
            return []
    except (TypeError, ValueError):
        pass
    if isinstance(vals, (list, tuple, np.ndarray, set)):
        return [str(v) for v in vals if v is not None and str(v) not in ("", "nan")]
    s = str(vals)
    return [] if s in ("", "nan", "None") else [s]


def genes_by_qtl_modality(phenotype_id: str):
    eq, sq = set(), set()
    for part in str(phenotype_id).split(";"):
        stems = _ensg_stem.findall(part)
        if not stems:
            continue
        g = stems[-1]
        if "expression_" in part:
            eq.add(g)
        elif "splicing_" in part:
            sq.add(g)
    return eq, sq


def normalize_variant_id(variant_id: str) -> str:
    s = str(variant_id)
    sep = ":" if ":" in s else "_"
    parts = s.split(sep)
    if len(parts) >= 4:
        return f"{parts[0]}:{parts[1]}:{parts[2]}:{parts[3]}"
    return s


def frac_tick_label(frac: float) -> str:
    pct = 100.0 * frac
    if abs(pct - round(pct)) < 0.05:
        return f"{int(round(pct))}%"
    return f"{pct:.1f}%"


### Compute — TSS map


In [ ]:
# 0-based strand-aware gene-boundary TSS; v47 overrides v39.
attr_re = re.compile(r'(\w+)\s+"([^"]+)"')
tss_by_key = {}
for gtf_path in (tss_gtf_v39, tss_gtf_v47):
    if not gtf_path.is_file():
        raise FileNotFoundError(gtf_path)
    n_genes = 0
    with open(gtf_path) as fh:
        for line in tqdm(fh, desc=f"TSS GTF {gtf_path.name}"):
            if not line or line.startswith("#"):
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 9 or fields[2] != "gene":
                continue
            chrom, start1, end1, strand = fields[0], int(fields[3]), int(fields[4]), fields[6]
            gid = dict(attr_re.findall(fields[8])).get("gene_id", "")
            if not gid:
                continue
            if strand == "+":
                tss0 = start1 - 1
            elif strand == "-":
                tss0 = end1 - 1
            else:
                raise ValueError(f"Unexpected strand {strand!r} for {gid} in {gtf_path}")
            tss_by_key[gid] = (chrom, tss0)
            tss_by_key[gid.split(".")[0]] = (chrom, tss0)
            n_genes += 1
    print(f"{gtf_path.name}: {n_genes:,} gene features")
tss_rows = [(gid, chrom, tss) for gid, (chrom, tss) in tss_by_key.items()]
tss_map = {gid: (chrom, int(tss)) for gid, chrom, tss in tss_rows}
print(f"TSS map keys: {len(tss_map):,}")


In [ ]:
print(f"TSS map ready ({len(tss_map):,} keys)")


### Compute — within-1 Mb fraction


In [ ]:


def build_coloc_outside_table(lead_files):
    rows = []
    for lead_path in tqdm(lead_files, desc="tissues"):
        tissue = Path(lead_path).name.split(".")[0]
        leads = pd.read_parquet(lead_path)
        leads = leads[leads["cos_npc"] > cos_npc_cut].copy()
        leads["has_gwas_1e5"] = [
            has_gwas_gws(pid, csmax, cut=gwas_neglog10p_1e5)
            for pid, csmax in zip(leads["phenotype_id"], leads["cs_max_neg_log10_p_value"])
        ]
        leads["is_gws"] = [
            has_gwas_gws(pid, csmax, cut=gws_neglog10p)
            for pid, csmax in zip(leads["phenotype_id"], leads["cs_max_neg_log10_p_value"])
        ]
        leads = leads[leads["has_gwas_1e5"] & (leads["n_unique_genes"] > 0)]
        if leads.empty:
            continue

        rob_path = robust_parquet_for_leads(lead_path)
        wanted = set(leads["coloc_id"].astype(str))
        cs_map = load_cos_variant_list_map(rob_path, wanted)

        for _, r in leads.iterrows():
            genes = list(r["gene_ids"]) if r["gene_ids"] is not None else []
            if not genes:
                continue
            lead_vid = lead_variant_id(r)
            variants = cs_map.get(str(r["coloc_id"]), [lead_vid])
            outside_flags = []
            for g in genes:
                flag = gene_has_outside_cos(str(g), variants, tss_map)
                if flag is None:
                    continue
                outside_flags.append(bool(flag))
            if not outside_flags:
                continue
            n_genes = len(outside_flags)
            n_outside = int(sum(outside_flags))
            rows.append(
                {
                    "tissue": tissue,
                    "coloc_id": str(r["coloc_id"]),
                    "lead_variant_id": lead_vid,
                    "n_genes": n_genes,
                    "n_outside": n_outside,
                    "frac_outside": n_outside / n_genes,
                    "n_cs_variants": len(variants),
                    "is_gws": bool(r["is_gws"]),
                }
            )
    return pd.DataFrame(rows)

coloc_df = build_coloc_outside_table(lead_files_dist)


### Within-1 Mb fraction per CoS (panel b)

Per-tissue ColocBoost `joint_gwas` robust CoS leads from the full-distance (pre-cis-trim) run with CS-max GWAS p < 1×10⁻⁵ and ≥1 QTL gene; fraction of QTL genes with any CoS variant within TSS ± 1 Mb (`1 − frac_outside`). `is_gws` flags the GWS subset (p < 5.7×10⁻⁹).

In [ ]:
print(f"cis-window table: {len(coloc_df):,} rows")


### Compute — TSS–nearest CoS distances


In [ ]:


def min_signed_tss_distance(tss_chrom, tss, variant_ids):
    best = None
    for vid in variant_ids:
        vchrom, vpos = parse_variant_pos(vid)
        if not chrom_equal(vchrom, tss_chrom):
            continue
        signed = vpos - tss
        ad = abs(signed)
        if best is None or ad < best[0]:
            best = (ad, signed, str(vid), vpos)
    return best


def build_tss_min_cos_distance_table(lead_files_dist):
    rows = []
    for lead_path in tqdm(lead_files_dist, desc="TSS min-CoS distances"):
        tissue = Path(lead_path).name.split(".")[0]
        leads = pd.read_parquet(lead_path)
        leads = leads[leads["cos_npc"] > cos_npc_cut].copy()
        leads["has_gwas_1e5"] = [
            has_gwas_gws(pid, csmax, cut=gwas_neglog10p_1e5)
            for pid, csmax in zip(leads["phenotype_id"], leads["cs_max_neg_log10_p_value"])
        ]
        leads["is_gws"] = [
            has_gwas_gws(pid, csmax, cut=gws_neglog10p)
            for pid, csmax in zip(leads["phenotype_id"], leads["cs_max_neg_log10_p_value"])
        ]
        leads = leads[leads["has_gwas_1e5"] & (leads["n_unique_genes"] > 0)]
        if leads.empty:
            continue

        rob_path = robust_parquet_for_leads(lead_path)
        wanted = set(leads["coloc_id"].astype(str))
        cs_map = load_cos_variant_list_map(rob_path, wanted)

        for _, r in leads.iterrows():
            genes = list(r["gene_ids"]) if r["gene_ids"] is not None else []
            if not genes:
                continue
            lead_vid = lead_variant_id(r)
            variants = cs_map.get(str(r["coloc_id"]), [lead_vid])
            for g in genes:
                key = str(g) if str(g) in tss_map else str(g).split(".")[0]
                if key not in tss_map:
                    continue
                chrom, tss = tss_map[key]
                best = min_signed_tss_distance(chrom, tss, variants)
                if best is None:
                    continue
                abs_dist, signed, closest_vid, closest_pos = best
                rows.append(
                    {
                        "tissue": tissue,
                        "coloc_id": str(r["coloc_id"]),
                        "gene_id": str(g),
                        "lead_variant_id": lead_vid,
                        "closest_variant_id": closest_vid,
                        "tss": tss,
                        "closest_pos": closest_pos,
                        "tss_min_cos_distance": signed,
                        "abs_tss_min_cos_distance": abs_dist,
                        "outside_1mb": abs_dist > cis_bp,
                        "n_cs_variants": len(variants),
                        "is_gws": bool(r["is_gws"]),
                    }
                )
    return pd.DataFrame(rows)

gene_dist_df = build_tss_min_cos_distance_table(lead_files_dist)


### TSS–nearest CoS variant distances (panel a)

For each QTL gene in a p < 1×10⁻⁵ GWAS–QTL CoS (same leads as panel b), absolute distance from gene TSS to the nearest CoS variant (`min |variant_pos − TSS|`). `outside_1mb` is True iff no CoS variant lies within TSS ± 1 Mb.

In [ ]:
print(f"gene-distance table: {len(gene_dist_df):,} rows")


### Compute — SuSiE CS overlap


In [ ]:
susie_cs_cache = {}


def susie_phenotypes_from_coloc(tissue: str, phenotype_id: str):
    prefix_expr = f"{tissue}.expression_"
    prefix_spl = f"{tissue}.splicing_"
    for part in str(phenotype_id).split(";"):
        part = part.strip()
        if part.startswith(prefix_expr):
            yield "expression", part[len(prefix_expr):]
        elif part.startswith(prefix_spl):
            yield "splicing", part[len(prefix_spl):]


def load_susie_cs(tissue: str, kind: str) -> dict:
    key = (tissue, kind)
    if key in susie_cs_cache:
        return susie_cs_cache[key]
    if kind == "expression":
        path = eqtl_susie_dir / f"{tissue}.v11.eQTLs.SuSiE_summary.parquet"
    else:
        path = sqtl_susie_dir / f"{tissue}.v11.sQTLs.SuSiE_summary.parquet"
    if not path.is_file():
        raise FileNotFoundError(f"Missing SuSiE summary for QTL tissue: {path}")
    df = pd.read_parquet(path, columns=["phenotype_id", "cs_id", "variant_id"])
    out = (
        df.groupby("phenotype_id")["variant_id"]
        .apply(lambda s: {normalize_variant_id(v) for v in s.astype(str)})
        .to_dict()
    )
    susie_cs_cache[key] = out
    return out


def build_susie_overlap_table(lead_files):
    rows = []
    for lead_path in tqdm(lead_files, desc="SuSiE overlap"):
        tissue = Path(lead_path).name.split(".")[0]
        leads = pd.read_parquet(lead_path)
        leads = leads[leads["cos_npc"] > cos_npc_cut].copy()
        leads["has_gwas_1e5"] = [
            has_gwas_gws(pid, csmax, cut=gwas_neglog10p_1e5)
            for pid, csmax in zip(leads["phenotype_id"], leads["cs_max_neg_log10_p_value"])
        ]
        leads["is_gws"] = [
            has_gwas_gws(pid, csmax, cut=gws_neglog10p)
            for pid, csmax in zip(leads["phenotype_id"], leads["cs_max_neg_log10_p_value"])
        ]
        leads = leads[leads["has_gwas_1e5"] & (leads["n_unique_genes"] > 0)]
        if leads.empty:
            continue

        rob_path = robust_parquet_for_leads(lead_path)
        wanted = set(leads["coloc_id"].astype(str))
        rob = pd.read_parquet(rob_path)
        if "coloc_id" not in rob.columns:
            rob["coloc_id"] = rob["filtered_gene"].astype(str) + "_" + rob["cs_id"].astype(str)
        rob = rob[rob["coloc_id"].astype(str).isin(wanted)]
        cs_map = (
            rob.groupby("coloc_id")["variant_id"]
            .apply(lambda s: {normalize_variant_id(v) for v in s.astype(str)})
            .to_dict()
        )

        for _, r in leads.iterrows():
            lead_vid = lead_variant_id(r)
            cos_vars = cs_map.get(str(r["coloc_id"]), {normalize_variant_id(lead_vid)})
            phenos = list(susie_phenotypes_from_coloc(tissue, r["phenotype_id"]))
            if not phenos:
                continue
            n_in = 0
            for kind, pheno_id in phenos:
                susie_cs = load_susie_cs(tissue, kind).get(pheno_id, set())
                if cos_vars & susie_cs:
                    n_in += 1
            rows.append(
                {
                    "tissue": tissue,
                    "coloc_id": str(r["coloc_id"]),
                    "lead_variant_id": lead_vid,
                    "n_phenotypes": len(phenos),
                    "n_in_susie_cs": n_in,
                    "frac_in_susie_cs": n_in / len(phenos),
                    "n_cs_variants": len(cos_vars),
                    "is_gws": bool(r["is_gws"]),
                }
            )
    return pd.DataFrame(rows)

susie_cs_cache.clear()
susie_df = build_susie_overlap_table(lead_files)


### SuSiE CS overlap per CoS (panel c)

Per-tissue ColocBoost `joint_gwas` robust CoS leads from the cis-trimmed run, p < 1×10⁻⁵ GWAS. For each QTL phenotype in the CoS, overlap = any CoS variant in that phenotype's tissue SuSiE 95% credible set. Expression and splicing for the same gene are separate phenotypes.

In [ ]:
print(
    f"  frac>0: {(susie_df['frac_in_susie_cs'] > 0).sum():,} "
    f"({100 * (susie_df['frac_in_susie_cs'] > 0).mean():.1f}%); "
    f"frac==1: {(susie_df['frac_in_susie_cs'] == 1).sum():,} "
    f"({100 * (susie_df['frac_in_susie_cs'] == 1).mean():.1f}%)"
)


### GENCODE classification


In [ ]:
gencode_genes = pd.read_table(gencode_class, low_memory=False)
gencode_genes = gencode_genes[gencode_genes["gene_type_grouped"] != "artifact"]
print(f"GENCODE genes (non-artifact): {len(gencode_genes):,}")


### Cross-tissue locus summary and long


In [ ]:
locus_summary = pd.read_parquet(cross_summary)
locus_long = pd.read_parquet(cross_long)
need = [
    "has_gwas_gws", "n_gwas_gws", "gwas_gws_traits", "qtl_label",
    "robust_qtl_types", "robust_gene_ids", "relaxed_qtl_types", "relaxed_gene_ids",
]
missing = [c for c in need if c not in locus_summary.columns]
if missing:
    raise KeyError(f"Summary missing {missing}. Re-run cross-tissue merge.")

locus_summary["has_gwas_gws"] = locus_summary["has_gwas_gws"].fillna(False).astype(bool)
locus_summary["qtl_label_raw"] = locus_summary["qtl_label"].astype(str)
locus_summary["qtl_label"] = (
    locus_summary["qtl_label_raw"].map(qtl_label_for_figure).fillna(locus_summary["qtl_label_raw"])
)
print(
    f"Loci: {len(locus_summary):,}; members: {len(locus_long):,}; "
    f"GWS: {locus_summary['has_gwas_gws'].sum():,}"
)


### Ascertainment tables (expression, junctions, e/sGenes)


In [ ]:
if not median_counts_file.exists():
    raise FileNotFoundError(f"Missing median counts: {median_counts_file}")
tissue_medians = pd.read_table(median_counts_file, index_col=0)
if not any(c in qtl_tissue_ids for c in tissue_medians.columns):
    tissue_medians.columns = [
        c.replace("(", "").replace(")", "").replace(" - ", "_").replace(" ", "_")
        for c in tissue_medians.columns
    ]
print(f"median counts: {tissue_medians.shape[0]:,} genes × {tissue_medians.shape[1]} tissues")

if not n_junctions_file.exists():
    raise FileNotFoundError(f"Missing n_junctions: {n_junctions_file}")
n_junctions = pd.read_parquet(n_junctions_file)
print(f"n_junctions: {len(n_junctions):,} rows")

if not egene_gt_file.exists():
    raise FileNotFoundError(f"Missing eGene GT: {egene_gt_file}")
if not sgene_gt_file.exists():
    raise FileNotFoundError(f"Missing sGene GT: {sgene_gt_file}")
egene_gt = pd.read_parquet(egene_gt_file)
sgene_gt = pd.read_parquet(sgene_gt_file)
print(f"eGene GT: {len(egene_gt):,}; sGene GT: {len(sgene_gt):,}")


### GWAS p < 1e-5 CoS filter (panels d–f)


In [ ]:
# Member-level p < 1e-5 (reuse has_gwas_gws with sub_sig cut)
locus_long_1e5 = locus_long[
    [
        has_gwas_gws(pid, csmax, cut=gwas_neglog10p_1e5)
        for pid, csmax in zip(locus_long["phenotype_id"], locus_long["cs_max_neg_log10_p_value"])
    ]
].copy()
print(
    f"locus_long members: all-GWAS={len(locus_long):,}; "
    f"p<1e-5={len(locus_long_1e5):,}; "
    f"GWS={int(locus_long['has_gwas_gws'].fillna(False).sum()):,}"
)

all_loci = locus_summary.copy()
n_no_qtl_all = int((all_loci["qtl_label"].astype(str) == "no_qtl").sum())
if n_no_qtl_all:
    all_loci = all_loci[all_loci["qtl_label"].astype(str) != "no_qtl"].copy()
loci_1e5_ids = set(locus_long_1e5["locus_id"].astype(str))
n_before = len(all_loci)
all_loci = all_loci[all_loci["locus_id"].astype(str).isin(loci_1e5_ids)].copy()
all_loci["gene_stems"] = all_loci["robust_gene_ids"].map(as_stem_set)
print(
    f"gene-coloc loci: all-GWAS (excl no_qtl)={n_before:,}; "
    f"p<1e-5={len(all_loci):,}; "
    f"GWS={int(locus_summary['has_gwas_gws'].sum()):,} "
    f"(dropped no_qtl={n_no_qtl_all:,})"
)


### Compute — coloc genes ascertainment


In [ ]:
all_locus_ids = set(all_loci["locus_id"].astype(str))
gene_has_coloc_eqtl, gene_has_coloc_sqtl = set(), set()
tissue_coloc_eqtl, tissue_coloc_sqtl = {}, {}

for r in tqdm(
    locus_long_1e5.itertuples(index=False), total=len(locus_long_1e5), desc="coloc modality"
):
    if str(r.locus_id) not in all_locus_ids:
        continue
    genes = set(as_stem_set(getattr(r, "genes", None)))
    if not genes:
        continue
    eq, sq = genes_by_qtl_modality(r.phenotype_id)
    eq &= genes
    sq &= genes
    if not eq and not sq:
        pt = set(as_str_list(r.phenotype_types)) - {"gwas"}
        if pt == {"eqtl"}:
            eq = genes
        elif pt == {"sqtl"}:
            sq = genes
    gene_has_coloc_eqtl |= eq
    gene_has_coloc_sqtl |= sq
    tissue = str(r.tissue) if r.tissue is not None else None
    if tissue and tissue != "nan":
        if eq:
            tissue_coloc_eqtl.setdefault(tissue, set()).update(eq)
        if sq:
            tissue_coloc_sqtl.setdefault(tissue, set()).update(sq)

print(
    f"p<1e-5 coloc genes: eQTL={len(gene_has_coloc_eqtl):,}, "
    f"sQTL={len(gene_has_coloc_sqtl):,}; "
    f"tissues with e/s coloc: {len(tissue_coloc_eqtl)}/{len(tissue_coloc_sqtl)}"
)

coloc_gene_meta = gencode_genes[[
    "gene_id_stable", "gene_id_v47", "gene_type_grouped", "in_v47_full",
]].copy()
coloc_gene_meta = coloc_gene_meta[coloc_gene_meta["in_v47_full"]].copy()
coloc_gene_meta = coloc_gene_meta[
    coloc_gene_meta["gene_type_grouped"].isin(gene_type_order)
].copy()
coloc_gene_meta = coloc_gene_meta.drop_duplicates("gene_id_stable")

qtl_cols = [c for c in tissue_medians.columns if c in qtl_tissue_ids]
_missing = sorted(qtl_tissue_ids - set(qtl_cols))
if _missing:
    raise ValueError(
        f"Median counts missing {len(_missing)}/{len(qtl_tissue_ids)} QTL tissues: {_missing}"
    )
max_med = tissue_medians[qtl_cols].max(axis=1).rename("max_median_counts")
gene_expr_stats = max_med.rename_axis("gene_id").reset_index()
gene_expr_stats["gene_id_stable"] = gene_expr_stats["gene_id"].str.split(".").str[0]
gene_expr_stats = (
    gene_expr_stats.groupby("gene_id_stable", as_index=False)["max_median_counts"].max()
)

n_junctions_local = n_junctions
if "gene_id_stable" not in n_junctions_local.columns and "gene_id" in n_junctions_local.columns:
    n_junctions_local = n_junctions_local.copy()
    n_junctions_local["gene_id_stable"] = n_junctions_local["gene_id"].astype(str).str.split(".").str[0]
junc_gene = (
    n_junctions_local.groupby("gene_id_stable", as_index=False)["n_junctions"]
    .max()
    .rename(columns={"n_junctions": "max_n_junctions"})
)

egene_local = egene_gt
sgene_local = sgene_gt
if "gene_id_stable" not in egene_local.columns and "gene_id" in egene_local.columns:
    egene_local = egene_local.copy()
    egene_local["gene_id_stable"] = egene_local["gene_id"].astype(str).str.split(".").str[0]
if "gene_id_stable" not in sgene_local.columns and "gene_id" in sgene_local.columns:
    sgene_local = sgene_local.copy()
    sgene_local["gene_id_stable"] = sgene_local["gene_id"].astype(str).str.split(".").str[0]
n_eqtl_tested = egene_local.groupby("gene_id_stable")["tissue"].nunique().rename("n_tissues_tested_eqtl")
n_sqtl_tested = sgene_local.groupby("gene_id_stable")["tissue"].nunique().rename("n_tissues_tested_sqtl")

coloc_genes = coloc_gene_meta.merge(gene_expr_stats, on="gene_id_stable", how="left")
coloc_genes = coloc_genes.merge(junc_gene, on="gene_id_stable", how="left")
coloc_genes["max_n_junctions"] = coloc_genes["max_n_junctions"].fillna(0)
coloc_genes["n_tissues_tested_eqtl"] = (
    coloc_genes["gene_id_stable"].map(n_eqtl_tested).fillna(0).astype(int)
)
coloc_genes["n_tissues_tested_sqtl"] = (
    coloc_genes["gene_id_stable"].map(n_sqtl_tested).fillna(0).astype(int)
)
coloc_genes["is_coloc_eqtl"] = (
    coloc_genes["gene_id_stable"].isin(gene_has_coloc_eqtl).astype(int)
)
coloc_genes["is_coloc_sqtl"] = (
    coloc_genes["gene_id_stable"].isin(gene_has_coloc_sqtl).astype(int)
)
coloc_genes = coloc_genes[
    coloc_genes["gene_type_grouped"].isin(gene_type_order)
].copy()

tissue_rows = []
for tissue, genes in tissue_coloc_eqtl.items():
    tissue_rows.extend((tissue, g, "eqtl") for g in genes)
for tissue, genes in tissue_coloc_sqtl.items():
    tissue_rows.extend((tissue, g, "sqtl") for g in genes)
tissue_long = pd.DataFrame(tissue_rows, columns=["tissue", "gene_id_stable", "modality"])


### Gene-level coloc indicators and ascertainment covariates (panels d–f)

One row per GENCODE v47 lncRNA / protein-coding gene: coloc eQTL / sQTL indicator from cross-tissue GWAS p < 1×10⁻⁵ CoS members, max tissue median read count, max tested introns, and number of tissues eQTL/sQTL-tested.

In [ ]:
coloc_genes = coloc_genes.copy()
coloc_genes["gene_type_grouped"] = coloc_genes["gene_type_grouped"].replace(
    {"new_lncRNA": "novel_lncRNA"}
)
coloc_genes = coloc_genes[coloc_genes["gene_type_grouped"].isin(gene_type_order)].copy()
print(f"Coloc-gene table: {len(coloc_genes):,} genes")
print(coloc_genes["gene_type_grouped"].value_counts().reindex(gene_type_order).to_string())
print(
    f"Panels d–e genes: {len(coloc_genes):,}; "
    f"coloc eQTL rate={coloc_genes['is_coloc_eqtl'].mean():.3f}; "
    f"coloc sQTL rate={coloc_genes['is_coloc_sqtl'].mean():.3f}"
)


### Per-tissue coloc genes (panel d)

Long table of (tissue, gene, eQTL/sQTL) for genes in a GWAS p < 1×10⁻⁵ CoS in that tissue.

In [ ]:
print(f"Per-tissue coloc rows: {len(tissue_long):,}")
print(f"tissues with coloc eQTL: {len(tissue_coloc_eqtl):,}; sQTL: {len(tissue_coloc_sqtl):,}")


## Run analysis

### Per-tissue % summaries, modality counts, and binned P(coloc) curves

In [ ]:
def build_coloc_pct_summary(tissue_gene_sets, any_col):
    rows, hollows = [], []
    genes_bt = coloc_genes[["gene_id_stable", "gene_type_grouped", any_col]].copy()
    for idx, gt in enumerate(gene_type_order):
        sel = genes_bt["gene_type_grouped"].eq(gt)
        n = int(sel.sum())
        gids = set(genes_bt.loc[sel, "gene_id_stable"])
        for tissue, gset in tissue_gene_sets.items():
            rows.append((gt, tissue, 100 * len(gset & gids) / n if n else 0.0))
        hollows.append((gt, 100 * genes_bt.loc[sel, any_col].sum() / n if n else 0.0, idx))
    return pd.DataFrame(rows, columns=["gene_type_grouped", "tissue", "pct"]), hollows


def binned_prob(df, xcol, ycol, n_bins=n_prob_bins, min_n=min_bin_n, z=1.96):
    """equal-frequency bins of x; median x, mean(y), Wald counting-noise CI."""
    d = df[[xcol, ycol]].dropna().copy()
    d = d[np.isfinite(d[xcol])]
    if d.empty:
        return pd.DataFrame(columns=["x", "p", "n", "p_lo", "p_hi"])
    d["_rank"] = d[xcol].rank(method="first")
    d["_bin"] = pd.qcut(d["_rank"], q=n_bins, duplicates="drop")
    out = (
        d.groupby("_bin", observed=True)
        .agg(x=(xcol, "median"), p=(ycol, "mean"), n=(ycol, "size"))
        .reset_index(drop=True)
    )
    out = out[out["n"] >= min_n].copy()
    se = np.sqrt(out["p"] * (1.0 - out["p"]) / out["n"])
    out["p_lo"] = np.clip(out["p"] - z * se, 0.0, 1.0)
    out["p_hi"] = np.clip(out["p"] + z * se, 0.0, 1.0)
    return out


coloc_eqtl_pct_summary, coloc_eqtl_hollows = build_coloc_pct_summary(
    tissue_coloc_eqtl, "is_coloc_eqtl"
)
coloc_sqtl_pct_summary, coloc_sqtl_hollows = build_coloc_pct_summary(
    tissue_coloc_sqtl, "is_coloc_sqtl"
)

coloc_prob_specs = [
    ("max_median_counts", "is_coloc_eqtl", "P(coloc eQTL)", "Max tissue median reads"),
    ("max_median_counts", "is_coloc_sqtl", "P(coloc sQTL)", "Max tissue median reads"),
    ("max_n_junctions", "is_coloc_sqtl", "P(coloc sQTL)", "Max tested introns"),
    ("n_tissues_tested_eqtl", "is_coloc_eqtl", "P(coloc eQTL)", "N tissues tested (eQTL)"),
    ("n_tissues_tested_sqtl", "is_coloc_sqtl", "P(coloc sQTL)", "N tissues tested (sQTL)"),
]
coloc_prob_curves = {}
for gt in tqdm(gene_type_order, desc="binned coloc probs"):
    gt_df = coloc_genes[coloc_genes["gene_type_grouped"] == gt]
    for xcol, ycol, _, _ in coloc_prob_specs:
        key = (gt, xcol, ycol)
        coloc_prob_curves[key] = binned_prob(gt_df, xcol, ycol)
        curve = coloc_prob_curves[key]
        print(
            f"{gt} {ycol}~{xcol}: {len(curve)} bins, "
            f"n={int(curve['n'].sum()) if len(curve) else 0}"
        )

gene_modality_stacks = {}
print("Unique coloc genes by biotype × QTL modality (GWAS p < 1e-5 CoS):")
for key, lab in gene_modality_rows:
    sub = coloc_genes[coloc_genes["gene_type_grouped"] == key]
    eq_only = int(((sub["is_coloc_eqtl"] == 1) & (sub["is_coloc_sqtl"] == 0)).sum())
    both = int(((sub["is_coloc_eqtl"] == 1) & (sub["is_coloc_sqtl"] == 1)).sum())
    sq_only = int(((sub["is_coloc_eqtl"] == 0) & (sub["is_coloc_sqtl"] == 1)).sum())
    gene_modality_stacks[key] = {
        "eqtl": eq_only,
        "both": both,
        "sqtl": sq_only,
        "total": eq_only + both + sq_only,
    }
    s = gene_modality_stacks[key]
    print(
        f"  {lab.replace(chr(10), ' ')}: total={s['total']:,} "
        f"eQTL={s['eqtl']:,} both={s['both']:,} sQTL={s['sqtl']:,}"
    )

## Make figure

In [ ]:
def _zero_one_hist_counts(values, n_bins=10):
    values = np.asarray(values, dtype=float)
    zero = values == 0
    one = values == 100
    mid = values[(~zero) & (~one)]
    hist, edges = np.histogram(mid, bins=n_bins, range=(0, 100))
    return int(zero.sum()), hist, edges, int(one.sum())


def plot_zero_one_overlay(ax, values, gws_mask, n_bins=10, edge_width=8):
    """Light bars = all values; dark bars = GWS subset (same bins, drawn on top)."""
    values = np.asarray(values, dtype=float)
    gws_mask = np.asarray(gws_mask, dtype=bool)
    n0, hist, edges, n1 = _zero_one_hist_counts(values, n_bins=n_bins)
    n0_g, hist_g, _, n1_g = _zero_one_hist_counts(values[gws_mask], n_bins=n_bins)
    bin_width = edges[1] - edges[0]
    ax.bar([0], [n0], color=bar_color, width=edge_width, edgecolor="none")
    ax.bar([0], [n0_g], color=gws_color, width=edge_width, edgecolor="none")
    for left, val, val_g in zip(edges[1:-1], hist, hist_g):
        ax.bar(left, val, color=bar_color, width=bin_width * 0.92, edgecolor="none")
        ax.bar(left, val_g, color=gws_color, width=bin_width * 0.92, edgecolor="none")
    ax.bar(100, n1, color=bar_color, width=edge_width, edgecolor="none")
    ax.bar(100, n1_g, color=gws_color, width=edge_width, edgecolor="none")
    xticks = [0] + list(edges[1:-1]) + [100]
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{int(v)}" for v in xticks], rotation=0, ha="center")


fig = plt.figure(figsize=(fig_width, fig_height), dpi=fig_dpi)
gs = fig.add_gridspec(2, 1, height_ratios=[1.0, 1.85], hspace=0.45)

gs_top = gs[0].subgridspec(1, 3, width_ratios=[1, 1, 1], wspace=0.40)
ax_a = fig.add_subplot(gs_top[0, 0])
ax_b = fig.add_subplot(gs_top[0, 1])
ax_c = fig.add_subplot(gs_top[0, 2])

# Panel a — absolute TSS → nearest CoS variant distances (p<1e-5 only)
dist_mb = gene_dist_df["abs_tss_min_cos_distance"].to_numpy(dtype=float) / 1e6
counts_a, edges_a = np.histogram(dist_mb, bins=50)
widths_a = np.diff(edges_a)
ax_a.bar(
    edges_a[:-1], counts_a, width=widths_a, align="edge",
    color=bar_color, edgecolor="none",
)
ax_a.axvline(1, color="0.4", lw=0.8, ls="--")
ax_a.set_yscale("log")
x_max = 10 * np.ceil(np.nanmax(dist_mb) / 10)
ax_a.set_xlim(0, x_max)
ax_a.set_xticks(np.arange(0, x_max + 1e-9, 10))
ax_a.set_xlabel("|TSS–nearest CoS| dist. (Mb)")
ax_a.set_ylabel("Genes")
ax_a.spines["top"].set_visible(False)
ax_a.spines["right"].set_visible(False)
n_dist = len(dist_mb)
pct_within_a = 100.0 * np.sum(dist_mb <= 1) / n_dist
pct_outside_a = 100.0 * np.sum(dist_mb > 1) / n_dist
_xlim = ax_a.get_xlim()
_span = _xlim[1] - _xlim[0]
_ann_x = [0.5, max((1 + _xlim[1]) / 2, 1 + 0.22 * _span)]
_ann_ha = ["center", "center"]
if _ann_x[1] - 1 < 0.18 * _span:
    _ann_x[1] = _xlim[1] - 0.02 * _span
    _ann_ha[1] = "right"
_trans = ax_a.get_xaxis_transform()
for x, pct, ha in zip(_ann_x, [pct_within_a, pct_outside_a], _ann_ha):
    ax_a.text(
        x, 0.98, f"{pct:.1f}%", transform=_trans, ha=ha, va="bottom",
        fontsize=6, clip_on=False,
    )

# Panel b — % of QTL genes within TSS ± 1 Mb
pct_within = 100.0 * (1.0 - coloc_df["frac_outside"].to_numpy(dtype=float))
gws_b = coloc_df["is_gws"].to_numpy(dtype=bool)
plot_zero_one_overlay(ax_b, pct_within, gws_b)
ax_b.set_ylabel("GWAS-QTL CoS")
ax_b.set_xlabel("Phenotypes within 1 Mb (%)")
ax_b.spines["top"].set_visible(False)
ax_b.spines["right"].set_visible(False)

# Panel c — SuSiE CS overlap
pct_susie = 100.0 * susie_df["frac_in_susie_cs"].to_numpy(dtype=float)
gws_c = susie_df["is_gws"].to_numpy(dtype=bool)
plot_zero_one_overlay(ax_c, pct_susie, gws_c)
ax_c.set_xlabel("CoS var in a SuSiE CS (%)")
ax_c.set_ylabel("GWAS-QTL CoS")
ax_c.spines["top"].set_visible(False)
ax_c.spines["right"].set_visible(False)

_legend_handles = [
    plt.Rectangle((0, 0), 1, 1, facecolor=bar_color, edgecolor="none", label=r"$p < 1\times10^{-5}$"),
    plt.Rectangle((0, 0), 1, 1, facecolor=gws_color, edgecolor="none", label=r"GWS ($p < 5.7\times10^{-9}$)"),
]
for _ax in (ax_b, ax_c):
    _ax.legend(handles=_legend_handles, frameon=False, loc="upper center", fontsize=5)

gs_ef = gs[1].subgridspec(2, 1, height_ratios=[0.55, 1.35], hspace=0.55)

gs_e = gs_ef[0].subgridspec(1, 3, width_ratios=[1, 1, 1], wspace=0.40)
axes_e = [fig.add_subplot(gs_e[0, j]) for j in range(2)]
ax_mod = fig.add_subplot(gs_e[0, 2])
for ax, summary, hollows, xlabel in zip(
    axes_e,
    [coloc_eqtl_pct_summary, coloc_sqtl_pct_summary],
    [coloc_eqtl_hollows, coloc_sqtl_hollows],
    ["% eQTL-GWAS CoS", "% sQTL-GWAS CoS"],
):
    for i in range(len(gene_type_order)):
        if i % 2 == 0:
            ax.axhspan(i - 0.5, i + 0.5, color="#fafafa", zorder=0)
    for gt, pct_any, y in hollows:
        ax.barh(
            y, pct_any, left=0, height=0.8,
            edgecolor=gene_type_palette[gt], facecolor=(0, 0, 0, 0),
            linewidth=0.8, zorder=1,
        )
    sns.stripplot(
        data=summary,
        y="gene_type_grouped",
        x="pct",
        order=gene_type_order,
        hue="tissue",
        palette=tissue_color,
        dodge=False,
        size=1.5,
        linewidth=0,
        alpha=0.9,
        zorder=2,
        ax=ax,
        orient="h",
        legend=False,
        jitter=0.2,
    )
    ax.set_yticks(range(len(gene_type_order)))
    ax.set_yticklabels([gene_type_pretty[g] for g in gene_type_order], fontsize=4.5)
    ax.tick_params(axis="y", left=False, labelleft=True)
    ax.set_ylabel("")
    ax.set_xlabel(xlabel)
    ax.set_xlim(-0.5, 32.5)
    ax.set_xticks([0, 10, 20, 30])
    ax.set_xticklabels(["0", "10", "20", "30"])
    ax.tick_params(axis="x", labelbottom=True)
    sns.despine(ax=ax, left=True, bottom=False)

# Panel e — unique coloc genes × eQTL / both / sQTL by biotype
edge_lw = 0.7
n_bio = len(gene_modality_rows)
n_mod = len(gene_modality_stack)
gene_bar_w = 0.18
gene_offsets = (np.arange(n_mod) - (n_mod - 1) / 2) * gene_bar_w
gene_ymax = 1
for i, (key, _lab) in enumerate(gene_modality_rows):
    s = gene_modality_stacks[key]
    for mod, dx in zip(gene_modality_stack, gene_offsets):
        h = s[mod]
        if h <= 0:
            continue
        ax_mod.bar(
            i + dx, h, width=gene_bar_w * 0.9,
            color=gene_modality_color[mod], edgecolor="black", linewidth=edge_lw,
        )
        ax_mod.annotate(
            f"{h:,}",
            xy=(i + dx, h),
            xytext=(0, 2),
            textcoords="offset points",
            va="bottom", ha="center",
            fontsize=3.5, rotation=90,
        )
        gene_ymax = max(gene_ymax, h)
ax_mod.set_xticks(range(n_bio))
ax_mod.set_xticklabels([lab for _, lab in gene_modality_rows])
ax_mod.set_xlabel("")
ax_mod.set_ylabel("Genes w/\nGWAS CoS")
ax_mod.set_yscale("log")
ax_mod.set_ylim(10, gene_ymax * 4.5)
ax_mod.set_yticks([10, 100, 1000, 10000])
ax_mod.set_yticklabels(["10", "100", "1k", "10k"])
ax_mod.set_xlim(-0.55, n_bio - 0.45)
ax_mod.tick_params(axis="x", length=0, pad=2)
ax_mod.spines["top"].set_visible(False)
ax_mod.spines["right"].set_visible(False)
ax_mod.legend(
    handles=[
        Patch(facecolor=gene_modality_color[m], edgecolor="black", linewidth=edge_lw,
              label=gene_modality_label[m])
        for m in gene_modality_stack
    ],
    loc="lower center", bbox_to_anchor=(0.5, 1.05),
    frameon=False, ncol=3, handletextpad=0.4, columnspacing=0.9, borderaxespad=0.0,
    fontsize=4,
)

gs_f_wrap = gs_ef[1].subgridspec(2, 1, height_ratios=[0.14, 1.0], hspace=0.1)
ax_f_leg = fig.add_subplot(gs_f_wrap[0])
ax_f_leg.set_axis_off()
gs_f = gs_f_wrap[1].subgridspec(1, 5, wspace=0.55)
axes_f = [fig.add_subplot(gs_f[0, j]) for j in range(5)]
for j, (xcol, ycol, ylabel, xlabel) in enumerate(coloc_prob_specs):
    ax = axes_f[j]
    for gt in gene_type_order:
        curve = coloc_prob_curves[(gt, xcol, ycol)]
        if curve.empty:
            continue
        xvals = curve["x"].to_numpy(dtype=float)
        if xcol == "max_median_counts":
            xvals = np.clip(xvals, 1e-1, None)
        color = gene_type_palette[gt]
        p_lo = curve["p_lo"].to_numpy(dtype=float)
        p_hi = curve["p_hi"].to_numpy(dtype=float)
        ax.fill_between(
            xvals, p_lo, p_hi,
            color=color, alpha=0.22, linewidth=0, zorder=1,
        )
        ax.plot(xvals, p_lo, color=color, lw=0.8, alpha=0.7, zorder=2)
        ax.plot(xvals, p_hi, color=color, lw=0.8, alpha=0.7, zorder=2)
        ax.plot(
            xvals, curve["p"].to_numpy(dtype=float),
            "o-", color=color, lw=0.8, markersize=2.5,
            alpha=0.95, zorder=3, label=gene_type_pretty[gt],
        )
    if xcol == "max_median_counts":
        ax.set_xscale("log")
    else:
        ax.set_xlim(left=-0.5)
    ax.tick_params(axis="both", labelsize=5)
    ax.set_ylim(0, 1)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

handles_f = [
    Line2D(
        [0], [0], color=gene_type_palette[gt], marker="o", markersize=3,
        lw=0.8, label=gene_type_pretty[gt],
    )
    for gt in gene_type_order
]
ax_f_leg.legend(
    handles=handles_f, loc="center", ncol=len(gene_type_order),
    frameon=False, fontsize=5,
)

# Panel letters — left of outermost y-label; a/d/f share one left x
fig.canvas.draw()
_renderer = fig.canvas.get_renderer()
_inv = fig.transFigure.inverted()


def _left_of_ylabels(ax, pad=0.012, fallback_dx=-0.35):
    artists = [t for t in ax.get_yticklabels() if t.get_visible() and t.get_text()]
    if ax.yaxis.label.get_text():
        artists.append(ax.yaxis.label)
    if not artists:
        return _inv.transform(ax.transAxes.transform((fallback_dx, 0)))[0]
    return min(a.get_window_extent(renderer=_renderer).transformed(_inv).x0 for a in artists) - pad


_label_axes = [
    (ax_a, "a"), (ax_b, "b"), (ax_c, "c"),
    (axes_e[0], "d"), (ax_mod, "e"), (axes_f[0], "f"),
]
_label_pos = {
    label: (
        _left_of_ylabels(ax),
        _inv.transform(ax.transAxes.transform((0, 1.08)))[1],
    )
    for ax, label in _label_axes
}
_x_adf = min(_label_pos[lab][0] for lab in ("a", "d", "f"))
for label in ("a", "b", "c", "d", "e", "f"):
    _x, _y = _label_pos[label]
    if label in ("a", "d", "f"):
        _x = _x_adf
    fig.text(_x, _y, label, fontweight="bold", va="bottom", ha="right", fontsize=9)

plt.show()

## Save figure

In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_pdf, bbox_inches="tight")
fig.savefig(fig_png, bbox_inches="tight")
print(f"Wrote {fig_pdf}")
print(f"Wrote {fig_png}")

## Print stats

In [ ]:
novel_modality = gene_modality_stacks["novel_lncRNA"]
print(
    f"Novel lncRNA modality: {novel_modality['eqtl']:,} expression only, "
    f"{novel_modality['sqtl']:,} splicing only, {novel_modality['both']:,} both"
)

_dist_mb = gene_dist_df["abs_tss_min_cos_distance"].to_numpy(dtype=float) / 1e6
print(
    f"Nearest CoS variant within/outside cis window: "
    f"{100 * np.mean(_dist_mb <= 1):.1f}% / "
    f"{100 * np.mean(_dist_mb > 1):.1f}%"
)

print(f"CoS with all QTL genes outside cis window: {100 * np.mean(coloc_df['frac_outside'] == 1):.1f}%")
print(f"CoS with some SuSiE overlap: {100 * np.mean(susie_df['frac_in_susie_cs'] > 0):.1f}%")

novel_genes = coloc_genes[coloc_genes["gene_type_grouped"] == "novel_lncRNA"]
print(
    f"Novel lncRNAs with GWAS-eQTL CoS: {100 * novel_genes['is_coloc_eqtl'].mean():.1f}%; "
    f"GWAS-sQTL CoS: {100 * novel_genes['is_coloc_sqtl'].mean():.1f}%"
)
